In [6]:
# 1
!pip -q install umap-learn
import json, re, unicodedata, warnings, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import CCA
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import mutual_info_regression
import umap
warnings.filterwarnings('ignore')

K = 3
NN = 10
SEEDS = [0, 1, 2, 3, 4]
TRAIN, TEST = 2016, (2017, 2022)
SUBS = {'2017_19': (2017, 2019), '2020_22': (2020, 2022)}
OUT = Path('cp3_out'); OUT.mkdir(exist_ok=True)

In [3]:
# 2
norm = lambda s: re.sub(r'[^a-z]', '', unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower())
ALIAS = {'dimbovita': 'dambovita', 'vilcea': 'valcea', 'bucuresti': 'ilfov'}

X = pd.read_csv('X_judet_luna.csv')
feat = ['ndvi', 'ndvi_anom', 'spei_1', 'spei_3', 'spei_6']
W = X.pivot_table(index=['judet', 'an'], columns='luna', values=feat)
W.columns = [f'{a}_m{m}' for a, m in W.columns]
W = W.reset_index()
W['k'] = W.judet.map(lambda s: ALIAS.get(norm(s), norm(s)))

VIEW = {'NDVI': [c for c in W.columns if c.startswith('ndvi')],
        'SPEI': [c for c in W.columns if c.startswith('spei')]}
VIEW['NDVI+SPEI'] = VIEW['NDVI'] + VIEW['SPEI']

ins = pd.read_csv('ins_agr110a.csv').dropna(subset=['kg_ha'])
ins['k'] = ins.judet.map(norm)

def yield_z(train_max):
    out = []
    for k, g in ins.groupby('k'):
        t = g[g.an <= train_max]
        if len(t) < 8:
            continue
        p = np.polyfit(t.an, t.kg_ha, 1)
        s = (t.kg_ha - np.polyval(p, t.an)).std(ddof=1)
        out.append(pd.DataFrame({'k': k, 'an': g.an, 'y': (g.kg_ha - np.polyval(p, g.an)) / s}))
    return pd.concat(out)

In [4]:
# 3
def dist_corr(A, B):
    return spearmanr(pdist(A), pdist(B))[0]

def knn(Z):
    return NearestNeighbors(n_neighbors=NN + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]

def overlap(A, B):
    return float(np.mean([len(set(a) & set(b)) / NN for a, b in zip(knn(A), knn(B))]))

def evaluate(B, Ztr, Zte, tr, te, subs):
    ytr, yte = tr.y.values, te.y.values
    mtr, mte = ~np.isnan(ytr), ~np.isnan(yte)
    ref = np.nanmean(ytr)
    model = Ridge(alpha=1.0).fit(Ztr[mtr], ytr[mtr])
    def downstream(mask):
        mk = mte & mask
        p = model.predict(Zte[mk])
        return np.sqrt(np.mean((yte[mk] - p) ** 2)), 1 - np.sum((yte[mk] - p) ** 2) / np.sum((yte[mk] - ref) ** 2)
    rmse, r2 = downstream(np.ones(len(te), bool))
    out = {'trust': trustworthiness(B, Zte, n_neighbors=NN), 'glob': dist_corr(B, Zte), 'rmse': rmse, 'r2': r2}
    for name, (a, b) in subs.items():
        mk = ((te.an >= a) & (te.an <= b)).values
        out['trust_' + name] = trustworthiness(B[mk], Zte[mk], n_neighbors=NN)
        out['r2_' + name] = downstream(mk)[1]
    return out

def run(train_max, test_min, test_max, k, subs):
    D = W[W.an <= test_max].merge(yield_z(train_max), how='left', on=['k', 'an'])
    tr = D[D.an <= train_max]
    te = D[(D.an >= test_min) & (D.an <= test_max)]
    rows = []
    for name, cols in VIEW.items():
        mu, sd = tr[cols].mean(), tr[cols].std()
        A = ((tr[cols] - mu) / sd).values
        B = ((te[cols] - mu) / sd).values
        kk = min(k, len(cols))
        raw = evaluate(B, A, B, tr, te, subs)
        raw.update(trust=1.0, glob=1.0)
        rows.append({'vista': name, 'metoda': 'fără reducere', 'seed': -1, 'k': len(cols), **raw})
        pca = PCA(n_components=kk).fit(A)
        Z = pca.transform(B)
        r = evaluate(B, pca.transform(A), Z, tr, te, subs)
        r.update(var=pca.explained_variance_ratio_.sum(),
                 recon=((B - pca.inverse_transform(Z)) ** 2).sum(1).mean() / ((B - A.mean(0)) ** 2).sum(1).mean())
        rows.append({'vista': name, 'metoda': 'PCA', 'seed': -1, 'k': kk, **r})
        for seed in SEEDS:
            um = umap.UMAP(n_components=kk, n_neighbors=15, min_dist=0.1, random_state=seed, n_jobs=1).fit(A)
            r = evaluate(B, um.embedding_, um.transform(B), tr, te, subs)
            rows.append({'vista': name, 'metoda': 'UMAP', 'seed': seed, 'k': kk, **r})
    return pd.DataFrame(rows), tr, te

In [5]:
# 4
main, tr, te = run(TRAIN, TEST[0], TEST[1], K, SUBS)
foldB, _, _ = run(2010, 2011, 2016, K, {})
main.to_csv(OUT/'main_k3.csv', index=False)
foldB.to_csv(OUT/'fold_2011_16_k3.csv', index=False)
print('train/test:', len(tr), len(te), '| cu randament:', int(tr.y.notna().sum()), int(te.y.notna().sum()))

cols = ['var', 'recon', 'trust', 'glob', 'rmse', 'r2']
tabel = main.groupby(['vista', 'metoda'])[cols].agg(['mean', 'std']).round(3)
tabel.to_csv(OUT/'tabel_principal.csv')
print(tabel.to_string())
sub = [c for c in main.columns if c.startswith('trust_') or c.startswith('r2_')]
main.groupby(['vista', 'metoda'])[sub].agg(['mean', 'std']).round(3).to_csv(OUT/'subperioade.csv')
foldB.groupby(['vista', 'metoda'])[['trust', 'glob', 'rmse', 'r2']].agg(['mean', 'std']).round(3).to_csv(OUT/'fold_2011_16.csv')

train/test: 656 246 | cu randament: 606 246
                           var      recon      trust          glob          rmse            r2       
                          mean std   mean std   mean    std   mean    std   mean    std   mean    std
vista     metoda                                                                                     
NDVI      PCA            0.762 NaN  0.299 NaN  0.925    NaN  0.878    NaN  1.375    NaN  0.215    NaN
          UMAP             NaN NaN    NaN NaN  0.931  0.009  0.668  0.010  1.459  0.022  0.115  0.027
          fără reducere    NaN NaN    NaN NaN  1.000    NaN  1.000    NaN  1.388    NaN  0.200    NaN
NDVI+SPEI PCA            0.677 NaN  0.492 NaN  0.893    NaN  0.784    NaN  1.446    NaN  0.132    NaN
          UMAP             NaN NaN    NaN NaN  0.898  0.006  0.347  0.029  1.552  0.009  0.000  0.011
          fără reducere    NaN NaN    NaN NaN  1.000    NaN  1.000    NaN  1.426    NaN  0.156    NaN
SPEI      PCA            0.787 NaN  0.

In [7]:
# 5
k8, _, _ = run(TRAIN, TEST[0], TEST[1], 8, {})
k8.groupby(['vista', 'metoda'])[['var', 'trust', 'glob', 'r2']].mean().round(3).to_csv(OUT/'tabel_k8.csv')
print(k8.groupby(['vista', 'metoda'])[['var', 'trust', 'glob', 'r2']].mean().round(3).to_string())

                           var  trust   glob     r2
vista     metoda                                   
NDVI      PCA            0.991  1.000  1.000  0.228
          UMAP             NaN  0.939  0.689  0.128
          fără reducere    NaN  1.000  1.000  0.200
NDVI+SPEI PCA            0.909  0.998  0.984  0.201
          UMAP             NaN  0.904  0.332 -0.022
          fără reducere    NaN  1.000  1.000  0.156
SPEI      PCA            0.994  1.000  1.000  0.022
          UMAP             NaN  0.915  0.373  0.054
          fără reducere    NaN  1.000  1.000  0.015


In [8]:
# 6
Dtr, Dte = W[W.an <= TRAIN], W[W.an > TRAIN]
V = {}
for n, c in VIEW.items():
    mu, sd = Dtr[c].mean(), Dtr[c].std()
    V[n] = (((Dtr[c] - mu) / sd).values, ((Dte[c] - mu) / sd).values)

rel = {'knn_ndvi_spei': overlap(V['NDVI'][1], V['SPEI'][1]),
       'knn_sansa': NN / (len(Dte) - 1),
       'knn_ndvi_comb': overlap(V['NDVI'][1], V['NDVI+SPEI'][1]),
       'knn_spei_comb': overlap(V['SPEI'][1], V['NDVI+SPEI'][1])}

pa, pb = PCA(5).fit(V['NDVI'][0]), PCA(5).fit(V['SPEI'][0])
cca = CCA(n_components=2, max_iter=2000).fit(pa.transform(V['NDVI'][0]), pb.transform(V['SPEI'][0]))
for nume, (a, b) in {'train': (V['NDVI'][0], V['SPEI'][0]), 'test': (V['NDVI'][1], V['SPEI'][1])}.items():
    u, v = cca.transform(pa.transform(a), pb.transform(b))
    rel['cca_' + nume] = [float(np.corrcoef(u[:, i], v[:, i])[0, 1]) for i in range(2)]

M = X.sort_values(['judet', 'an', 'luna']).copy()
for sc in ['spei_1', 'spei_3', 'spei_6']:
    for lag in [0, 1, 2]:
        M['x'] = M.groupby(['judet', 'an'])[sc].shift(lag)
        r = M.dropna(subset=['x', 'ndvi_anom']).groupby('judet').apply(lambda g: spearmanr(g.ndvi_anom, g.x)[0])
        rel[f'lag_{sc}_{lag}'] = [float(r.median()), float(r.quantile(.25)), float(r.quantile(.75))]

mi = mutual_info_regression(X[['spei_1', 'spei_3', 'spei_6']].values, X.ndvi_anom.values, random_state=0)
rel['mi'] = dict(zip(['spei_1', 'spei_3', 'spei_6'], map(float, mi)))
json.dump(rel, open(OUT/'relatii.json', 'w'), indent=2)
print(json.dumps(rel, indent=1))

{
 "knn_ndvi_spei": 0.3707317073170731,
 "knn_sansa": 0.04081632653061224,
 "knn_ndvi_comb": 0.582520325203252,
 "knn_spei_comb": 0.6434959349593495,
 "cca_train": [
  0.844924073131835,
  0.8040644606899338
 ],
 "cca_test": [
  0.755502527352221,
  0.5703235603746254
 ],
 "lag_spei_1_0": [
  0.44009746781944437,
  0.4157879083552044,
  0.4742946888386823
 ],
 "lag_spei_1_1": [
  0.5177783287875031,
  0.48640082952009556,
  0.5300498162883484
 ],
 "lag_spei_1_2": [
  0.10087703849811562,
  0.06060019020112008,
  0.17093445105843402
 ],
 "lag_spei_3_0": [
  0.5464772270718432,
  0.49542663042060386,
  0.5684857528971631
 ],
 "lag_spei_3_1": [
  0.35111692175912357,
  0.28482744629533624,
  0.4034217703942474
 ],
 "lag_spei_3_2": [
  0.105508787996196,
  -0.011095065337606989,
  0.18653798739037022
 ],
 "lag_spei_6_0": [
  0.4642818009631991,
  0.37932096027716755,
  0.5153428331411457
 ],
 "lag_spei_6_1": [
  0.3909158533011744,
  0.29900142010233755,
  0.45073146541036446
 ],
 "lag_spe

In [9]:
# 7
with zipfile.ZipFile('cp3_out.zip', 'w') as z:
    for p in OUT.iterdir():
        z.write(p, p.name)